## Read Me!

This notebook compares the predictions of the quadratic and quartic inflationary models with observational constraints on the scalar spectral index ($n_s$) and tensor-to-scalar ratio ($r$), using the SPA_BK and SPA_BK_DESI datasets.

Please note this notebook **requires** the model implementations and numerical calculations in the ```inflationary models.ipynb```, as well as the external MCMC chain files used to generate the observational constraints. 

The required chain files are not included in this repository. They can be obtained from the original repository accompanying [Balkenhol et al. 2025](https://arxiv.org/abs/2512.10613).

- [Original code and data repository](https://github.com/Lbalkenhol/r_ns_2025)


In [ ]:
from matplotlib.ticker import AutoMinorLocator
from getdist import plots, loadMCSamples
from getdist.gaussian_mixtures import GaussianND
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
chain_files = {
    # Chains for publication
    #SPA_BK": "chains/SPA_BK_streamlit/CLASS",
    #SPA_BK_DESI": "chains/SPA_BK_DESI_streamlit/CLASS",
    # Thinned chains for online app
    "SPA_BK": "chains/SPA_BK/CLASS",
    "SPA_BK_DESI": "chains/SPA_BK_DESI/CLASS",
}

chains = {}
bf_dicts = {}
for key, value in chain_files.items():
    chains[key] = loadMCSamples(value, settings={"ignore_rows": 0.0})# Burn-in has already been removed
    bf_dicts[key] = chains[key].getParamBestFitDict()

In [ ]:
# Style settings for different datasets


style_dict = {
    "SPA_BK": {
        "colour": sns.color_palette("colorblind")[0],
        "ls": "-",
        "lw": 0.75,
        "filled": True,
        "label": "SPA+BK",
    },
    "SPA_BK_DESI": {
        "colour": sns.color_palette("colorblind")[-1],
        "ls": "-",
        "lw": 1.5,
        "filled": False,
        "label": "SPA+BK+DESI",
    },
    "FC": {"colour": "k", "ls": "-", "lw": 1.5, "filled": False, "label": "CMB 2030s"},
    "FC_DESI": {
        "colour": "k",
        "ls": "--",
        "lw": 1.5,
        "filled": False,
        "label": "CMB 2030s+DESI",
    },
}

# Style settings for specific inflation models
# N* values can be single values or [N_min, N_max] ranges
alpha_unity_style_dict = {
    "Starobinsky $R^2$": {
        "color": "k",
        "edgecolor": "k",
        "lw": 1.2,
        "ms": 6,
        "s": 60,
        "marker": "s",
    },
    "Higgs": {
        "color": "w",
        "edgecolor": "k",
        "lw": 1.2,
        "ms": 7,
        "s": 70,
        "marker": "o",
    },
}

In [ ]:
# Create forecast chains preserving r-ns correlation from real data
cov = chains["SPA_BK"].cov(["r", "n_s"])
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
cov_scaled = np.diag([1e-3, 2e-3]) @ corr @ np.diag([1e-3, 2e-3])

mrg = chains["SPA_BK"].getMargeStats()
chains["FC"] = GaussianND([3e-3, mrg.parWithName("n_s").mean],
                                                    cov_scaled,
                                                    names=["r", "n_s"])

mrg_desi = chains["SPA_BK_DESI"].getMargeStats()
chains["FC_DESI"] = GaussianND([3e-3, mrg_desi.parWithName("n_s").mean],
                                                         cov_scaled,
                                                         names=["r", "n_s"])


In [ ]:
def r_ns(ns, p):
    """
    Calculate r as function of ns for power-law potential V ∝ φ^p.

    Parameters
    ----------
    ns : float or array
        Scalar spectral index
    p : float
        Power of the potential

    Returns
    -------
    float or array
        Tensor-to-scalar ratio r
    """
    return 8.0 * (1 - ns) * p / (2.0 + p)

def r_from_N_star_ns(N_star, ns):
    """
    Calculate r from N_star and ns for polynomial potentials.

    Parameters
    ----------
    N_star : float or array
        Number of e-folds
    ns : float or array
        Scalar spectral index

    Returns
    -------
    float or array
        Tensor-to-scalar ratio r
    """
    denom = 1 - 2 * N_star
    num = 2 * N_star * (8 * ns - 8) + 16
    return num / denom

def add_concave_convex_divide(ax, ns_range=None, **kwargs):
    """
    Add the concave/convex divide line (p=1 power law).

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to plot on
    ns_range : tuple or None
        (ns_min, ns_max) for the line. If None, uses (0.9, 1.1)
    **kwargs : dict
        Additional arguments passed to ax.plot()

    Returns
    -------
    matplotlib.lines.Line2D
        The plotted line
    """
    if ns_range is None:
        ns_range = (0.9, 1.1)

    default_kwargs = {"ls": "-", "color": "k", "lw": 1, "alpha": 0.8, "zorder": -5}
    default_kwargs.update(kwargs)

    ns = np.arange(ns_range[0], ns_range[1], 0.0001)
    line = ax.plot(ns, r_ns(ns, 1), **default_kwargs)
    return line[0]

def add_efold_shading_monomial(ax, N_range=(47, 57), ns_range=(0.96, 1.0), **kwargs):
    """
    Add shaded region showing e-fold range for monomial potentials V ∝ φⁿ.

    This shading is only mathematically valid for monomial (power-law) potentials.
    For other potential shapes, the N* constraints may differ.

    Parameters
    ----------
    ax : matplotlib.axes.Axes
        Axes to plot on
    N_range : tuple
        (N_min, N_max) e-fold range (default: 47, 57)
    ns_range : tuple
        (ns_min, ns_max) range for shading
    **kwargs : dict
        Additional arguments passed to ax.fill_between()

    Returns
    -------
    PolyCollection
        The shaded region
    """
    default_kwargs = {"lw": 1, "facecolor": "0.8", "edgecolor": "0.6", "zorder": -10}
    default_kwargs.update(kwargs)

    ns = np.linspace(ns_range[0], ns_range[1], 100)
    shading = ax.fill_between(
        ns,
        r_from_N_star_ns(N_range[0], ns),
        r_from_N_star_ns(N_range[1], ns),
        **default_kwargs,
    )
    return shading

In [ ]:
plt.close()

g = plots.get_single_plotter(width_inch=2.0*3.464, ratio=0.5)
g.settings.legend_frame = False
ax = plt.gca()

# Data constraints
all_dat = ["SPA_BK", "SPA_BK_DESI"]
g.plot_2d([chains[dat] for dat in all_dat],
          ["n_s", "r"],
          colors=[style_dict[dat]["colour"] for dat in all_dat],
          ls=[style_dict[dat]["ls"] for dat in all_dat],
          lws=[style_dict[dat]["lw"] for dat in all_dat],
          filled=[style_dict[dat]["filled"] for dat in all_dat])

ax = g.get_axes()
ax.plot(ns_region, r_region, color="r", lw=2, label="Quadratic inflation", zorder=10) #Insert model data in here
ax.legend()

# Add theory elements (publication N* range: 47-57)
add_efold_shading_monomial(ax, N_range=(50, 60), ns_range=(0.95, 1.0))
#add_concave_convex_divide(ax, ns_range=(0.95, 1.0))

# Legend
g.add_legend(["SPA+BK", "SPA+BK+DESI","Quadratic inflation"], fontsize=10)

# Axis setup
ax.set_ylim((0, 0.2))
ax.set_xlim((0.95, 1.0))
ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.set_ylabel("Tensor-to-scalar ratio $r$")
ax.set_xlabel("Scalar spectral index $n_s$")

plt.tight_layout()
plt.title('Tensor-to-scalar ratio $r$ vs Scalar spectral index $n_s$')
plt.savefig("ns_vs_k_observational_comparison_Quadratic.pdf")
plt.show()


